In [1]:
import torch
from datasets import load_dataset
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
import pandas as pd

/nesi/project/massey04342/home/mac/lib/python3.11/site-packages/trl/__init__.py:203: UserWarning: TRL currently supports vLLM versions: 0.10.2, 0.11.0, 0.11.1, 0.11.2. You have version 0.13.0 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():


In [2]:
MODEL_NAME = "Qwen/Qwen3-14B"
CSV_PATH = "data/lanka/sinhala/train.csv"   
TEXT_COLUMN = "prompt"
CACHE_DIR = "/nesi/nobackup/massey04342/models"

In [3]:
df = pd.read_csv(CSV_PATH)

df = df.rename(columns={
    "prompt": "prompt",      
    "output": "completion"   
})

df = df[["prompt", "completion"]]
dataset = Dataset.from_pandas(df)
dataset = dataset.remove_columns(
    [c for c in dataset.column_names if c not in ["prompt", "completion"]]
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
    cache_dir=CACHE_DIR
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

sft_config = SFTConfig(
    output_dir="./qwen3-math-sft",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    num_train_epochs=2,
    logging_steps=5,
    save_steps=200,
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    args=sft_config,
)

trainer.train()
trainer.save_model("./qwen3-math-sft/sinhala")